# Mobile Product Segmentation and Recommendation System

This notebook shows the exploratory data analysis, clustering (segmentation), and recommendation engine building for mobile products.

## Data Cleaning & Preparation

### Code from `src/db_config.py`

In [ ]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

def get_engine():
    db_user = os.getenv("DB_USER", "postgres")
    db_password = os.getenv("DB_PASSWORD")
    db_host = os.getenv("DB_HOST", "localhost")
    db_port = os.getenv("DB_PORT", "5432")
    db_name = os.getenv("DB_NAME", "product_segmentation")
    
    db_url = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
    return create_engine(db_url)


### Code from `src/data_ingestion.py`

In [ ]:
import pandas as pd
from db_config import get_engine
from pathlib import Path

# Build path to data folder relative to this script
DATA_DIR = Path(__file__).resolve().parent.parent / 'data'
csv_path = DATA_DIR / 'Mobile Reviews Sentiment null.csv'

# Load raw CSV
df = pd.read_csv(csv_path)

# Clean date column formatting before loading to Postgres
df['review_date'] = pd.to_datetime(df['review_date']).dt.date

# Connect and write to PostgreSQL
engine = get_engine()
df.to_sql('raw_mobile_reviews', engine, if_exists='replace', index=False)
print("Data ingestion complete!")

### Code from `src/data_cleaning.py`

In [ ]:
import pandas as pd
import numpy as np
from db_config import get_engine

print("Running Data Cleaning...")

# 1. Fetch data from PostgreSQL
engine = get_engine()
df = pd.read_sql('SELECT * FROM raw_mobile_reviews', engine)

# 2. Fix missing price_usd using model/brand medians (fall back to global median if still missing)
df['price_usd'] = df.groupby(['brand', 'model'])['price_usd'].transform(lambda x: x.fillna(x.median()))
df['price_usd'] = df['price_usd'].fillna(df['price_usd'].median())

# 3. Fix missing ratings using model averages (fall back to global mean if still missing)
df['rating'] = df.groupby('model')['rating'].transform(lambda x: x.fillna(x.mean()))
df['rating'] = df['rating'].fillna(df['rating'].mean())

# 4. Impute Missing Sentiment conditionally based on Rating
def impute_sentiment(row):
    if pd.isna(row['sentiment']):
        if row['rating'] >= 4.0: return 'Positive'
        elif row['rating'] == 3.0: return 'Neutral'
        else: return 'Negative'
    return row['sentiment']

df['sentiment'] = df.apply(impute_sentiment, axis=1)

# 5. Drop duplicate reviews if any exist
df.drop_duplicates(subset=['review_id'], inplace=True)

# 6. Save clean data back to PostgreSQL
df.to_sql('cleaned_mobile_reviews', engine, if_exists='replace', index=False)
print(f"Data cleaning complete! Cleaned reviews saved to 'cleaned_mobile_reviews' table ({len(df)} rows).")

### Code from `src/feature_engineering.py`

In [ ]:
import pandas as pd
from db_config import get_engine

print("Running Feature Engineering...")

# 1. Fetch cleaned data from PostgreSQL
engine = get_engine()
df = pd.read_sql('SELECT * FROM cleaned_mobile_reviews', engine)

# 2. Create a comprehensive composite specifications score
df['specs_average'] = df[['battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating']].mean(axis=1)

# 3. Group by brand and model to find product-level attributes
product_df = df.groupby(['brand', 'model']).agg(
    avg_price=('price_usd', 'mean'),
    avg_rating=('rating', 'mean'),
    avg_specs_score=('specs_average', 'mean'),
    total_reviews=('review_id', 'count'),
    positive_sentiment_ratio=('sentiment', lambda x: (x == 'Positive').sum() / len(x))
).reset_index()

# 4. Drop models with insufficient data if necessary (minimum 5 reviews)
product_df = product_df[product_df['total_reviews'] >= 5].reset_index(drop=True)

# 5. Save the aggregated product features back to PostgreSQL
product_df.to_sql('product_features', engine, if_exists='replace', index=False)

print(f"Feature engineering complete! Extracted features for {len(product_df)} unique phone models and saved to 'product_features' table.")

## Visualization & EDA

### Code from `src/eda.py`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from db_config import get_engine

print("Running Exploratory Data Analysis (EDA)...")

# 1. Fetch cleaned data from PostgreSQL
engine = get_engine()
df = pd.read_sql('SELECT * FROM cleaned_mobile_reviews', engine)

# 2. Correlation Heatmap for Specification Features
spec_cols = ['price_usd', 'rating', 'battery_life_rating', 'camera_rating', 'performance_rating', 'design_rating', 'display_rating']
plt.figure(figsize=(10, 6))
sns.heatmap(df[spec_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Specifications and Price')
plt.savefig('specifications_correlation.png', bbox_inches='tight')
plt.close()

print("EDA complete! Saved correlation heatmap to 'specifications_correlation.png'.")

## Model Performance & Results

### Code from `src/model_training.py`

In [ ]:
import pandas as pd
import pickle
from db_config import get_engine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pathlib import Path

print("Running Model Training...")

# 1. Fetch product features from PostgreSQL
engine = get_engine()
product_df = pd.read_sql('SELECT * FROM product_features', engine)

# 2. Select features for segmentation
features = ['avg_price', 'avg_rating', 'avg_specs_score', 'positive_sentiment_ratio']
X = product_df[features]

# 3. Create two scalers: one for recommendations (4 features) and one for quality (3 features)
scaler = StandardScaler()
scaler.fit(X)  # Fit on all 4 features (for compatibility with recommendation similarity)

# Quality scaler (3 features: rating, specs, sentiment ratio)
quality_features = ['avg_rating', 'avg_specs_score', 'positive_sentiment_ratio']
quality_scaler = StandardScaler()
X_quality_scaled = quality_scaler.fit_transform(product_df[quality_features])

# 4. Fit 2-cluster KMeans for quality segmentation (High vs Low Quality)
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
product_df['quality_cluster'] = kmeans.fit_predict(X_quality_scaled)

# Identify which cluster index is High Quality vs Low Quality
cluster_0_mean = product_df[product_df['quality_cluster'] == 0]['avg_rating'].mean()
cluster_1_mean = product_df[product_df['quality_cluster'] == 1]['avg_rating'].mean()
high_quality_cluster = 0 if cluster_0_mean > cluster_1_mean else 1

product_df['quality_label'] = product_df['quality_cluster'].apply(
    lambda x: 'High Quality' if x == high_quality_cluster else 'Low Quality'
)

# 5. Classify price tiers
def get_price_tier(price):
    if price >= 1000:
        return 'Premium'
    elif price >= 500:
        return 'Mid-Range'
    else:
        return 'Budget'

product_df['price_tier'] = product_df['avg_price'].apply(get_price_tier)

# Apply Hybrid Logic for final Business Persona Mapping
def get_hybrid_cluster_name(row):
    if row['quality_label'] == 'Low Quality':
        return 'Underperformers'
    else:
        if row['price_tier'] == 'Premium':
            return 'Premium Flagships'
        elif row['price_tier'] == 'Mid-Range':
            return 'Mid-Range Value'
        else:
            return 'Budget Workhorses'

product_df['cluster_name'] = product_df.apply(get_hybrid_cluster_name, axis=1)

# Maintain numerical 'cluster' mapping for reverse compatibility:
# 0: Budget Workhorses, 1: Underperformers, 2: Premium Flagships, 3: Mid-Range Value
CLUSTER_INT_MAPPING = {
    'Budget Workhorses': 0,
    'Underperformers': 1,
    'Premium Flagships': 2,
    'Mid-Range Value': 3
}
product_df['cluster'] = product_df['cluster_name'].map(CLUSTER_INT_MAPPING)

# Clean up helper columns before saving to keep database clean
product_df = product_df.drop(columns=['quality_cluster'])

# Show cluster profiles with business names
cluster_profiles = product_df.groupby('cluster_name')[features].mean()
print("Cluster Profiles (Mean values of features per cluster):")
print(cluster_profiles)

# 6. Save models/artifacts to disk using pickle
# Get the project root folder to save models and CSV
PROJECT_ROOT = Path(__file__).resolve().parent.parent

with open(PROJECT_ROOT / 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open(PROJECT_ROOT / 'kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)

# 7. Save clustered product data back to PostgreSQL and a CSV in the data folder
product_df.to_sql('segmented_products', engine, if_exists='replace', index=False)
product_df.to_csv(PROJECT_ROOT / 'data' / 'segmented_products.csv', index=False)

print("Model training complete! Scaler and KMeans models saved. Segmented products saved to database and CSV.")

### Code from `src/recommendation_engine.py`

In [ ]:
import pandas as pd
import pickle
from db_config import get_engine
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

# Get the project root folder
PROJECT_ROOT = Path(__file__).resolve().parent.parent

# 1. Fetch segmented products from PostgreSQL (fallback to CSV if DB is unavailable)
try:
    engine = get_engine()
    product_df = pd.read_sql('SELECT * FROM segmented_products', engine)
except Exception:
    # Fallback to local CSV
    product_df = pd.read_csv(PROJECT_ROOT / 'data' / 'segmented_products.csv')

# 2. Load the trained scaler to scale features
with open(PROJECT_ROOT / 'scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# 3. Scale features and compute the pairwise similarity matrix
features = ['avg_price', 'avg_rating', 'avg_specs_score', 'positive_sentiment_ratio']
X = product_df[features]
X_scaled = scaler.transform(X)

similarity_matrix = cosine_similarity(X_scaled)

def get_recommendations(model_name, top_n=5):
    # Find index of target phone model
    if model_name not in product_df['model'].values:
        return None
    
    target_row = product_df[product_df['model'] == model_name]
    idx = target_row.index[0]
    target_price_tier = target_row['price_tier'].values[0]
    
    # Get similarity scores for all phones with this phone
    sim_scores = list(enumerate(similarity_matrix[idx]))
    
    # Filter candidates:
    # 1. Exclude the selected phone itself.
    # 2. Prioritize High Quality models in the same price tier.
    filtered_sim_scores = []
    for other_idx, score in sim_scores:
        if other_idx != idx:
            other_row = product_df.loc[other_idx]
            if other_row['quality_label'] == 'High Quality' and other_row['price_tier'] == target_price_tier:
                filtered_sim_scores.append((other_idx, score))
    
    # Fallback: if we don't have enough recommendations in the same price tier,
    # pull other High Quality devices from adjacent/other price tiers.
    if len(filtered_sim_scores) < top_n:
        additional_scores = []
        for other_idx, score in sim_scores:
            if other_idx != idx and (other_idx, score) not in filtered_sim_scores:
                other_row = product_df.loc[other_idx]
                if other_row['quality_label'] == 'High Quality':
                    additional_scores.append((other_idx, score))
        
        # Sort additional recommendations by similarity score and append
        additional_scores = sorted(additional_scores, key=lambda x: x[1], reverse=True)
        filtered_sim_scores.extend(additional_scores[:(top_n - len(filtered_sim_scores))])
    
    # Sort by similarity score in descending order
    filtered_sim_scores = sorted(filtered_sim_scores, key=lambda x: x[1], reverse=True)[:top_n]
    
    # Extract phone details
    recommended_indices = [item[0] for item in filtered_sim_scores]
    
    if not recommended_indices:
        return pd.DataFrame(columns=['brand', 'model', 'avg_price', 'avg_rating', 'price_tier', 'cluster_name'])
        
    res_df = product_df.iloc[recommended_indices][['brand', 'model', 'avg_price', 'avg_rating', 'price_tier', 'cluster_name']].copy()
    
    # Custom sort order: Premium first, then Mid-Range, then Budget
    price_tier_order = {'Premium': 0, 'Mid-Range': 1, 'Budget': 2}
    res_df['price_tier_sort'] = res_df['price_tier'].map(price_tier_order)
    
    # Sort by price tier (ascending order of sort index) and then user rating (descending)
    res_df = res_df.sort_values(by=['price_tier_sort', 'avg_rating'], ascending=[True, False])
    res_df = res_df.drop(columns=['price_tier_sort'])
    
    return res_df